# DressApp — Phase 6 Eyes: Merge + GGUF + Push

Runs your Phase 6 LoRA fine-tune end-to-end:
1. Downloads base **`google/gemma-4-E2B-it`** (~10 GB)
2. Applies your LoRA adapter and **merges** into a single FP16 model
3. Pushes merged model to `Yoram-Jacobs/dressapp-eyes-pog-phase6` (private)
4. Converts to **GGUF F16**, quantises to **Q4_K_M**
5. Pushes GGUFs to `Yoram-Jacobs/dressapp-eyes-pog-phase6-gguf` (private)

**Runs on Colab T4 free tier (or any Linux box with ≥ 40 GB disk + 16 GB RAM).**
Total wall-clock: ~15–25 min. GPU is nice-to-have but **not required** — the merge works on CPU; only the GGUF convert benefits marginally from GPU.

### Prerequisites
1. Upload `adapter_model.safetensors`, `adapter_config.json`, `tokenizer.json`, `tokenizer_config.json`, `chat_template.jinja`, `processor_config.json`, `README.md` from your `pog_phase6_model/` folder into `/content/adapter/`.
2. Paste your **Hugging Face write token** (Settings → Access Tokens → New token → `write`) in the cell below.


In [ ]:
HF_TOKEN = "PASTE_YOUR_HF_WRITE_TOKEN_HERE"
HF_USER  = "Yoram-Jacobs"
BASE_MODEL = "google/gemma-4-E2B-it"
ADAPTER_DIR = "/content/adapter"
MERGED_DIR  = "/content/merged"
GGUF_DIR    = "/content/gguf"

REPO_MERGED = f"{HF_USER}/dressapp-eyes-pog-phase6"
REPO_GGUF   = f"{HF_USER}/dressapp-eyes-pog-phase6-gguf"

import os
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.makedirs(MERGED_DIR, exist_ok=True)
os.makedirs(GGUF_DIR,   exist_ok=True)

!mkdir -p {ADAPTER_DIR}
print("Upload your 7 adapter files to", ADAPTER_DIR, "before continuing.")

## 1 · Install pinned deps

In [ ]:
!pip install -q --upgrade "transformers>=4.57" "peft>=0.18" "accelerate>=1.0" "sentencepiece" "protobuf" "hf_transfer" "huggingface_hub>=0.24"

## 2 · Merge the LoRA into the base (FP16)

In [ ]:
import time, torch, gc
from transformers import AutoProcessor, AutoModelForImageTextToText, AutoModelForCausalLM
from peft import PeftModel

t0 = time.time()
print('Loading base model (downloading ~10 GB on first run)...')
try:
    model = AutoModelForImageTextToText.from_pretrained(BASE_MODEL, dtype=torch.float16, low_cpu_mem_usage=True)
    print('loaded via AutoModelForImageTextToText')
except Exception as e:
    print('ImageTextToText failed', type(e).__name__, '— falling back to CausalLM')
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.float16, low_cpu_mem_usage=True)

processor = AutoProcessor.from_pretrained(BASE_MODEL)
print(f'Base loaded in {time.time()-t0:.1f}s')

print('Applying LoRA adapter...')
model = PeftModel.from_pretrained(model, ADAPTER_DIR)
print('Merging + unloading...')
model = model.merge_and_unload()

print(f'Saving merged model to {MERGED_DIR}...')
model.save_pretrained(MERGED_DIR, safe_serialization=True, max_shard_size='4GB')
processor.save_pretrained(MERGED_DIR)
print(f'Done in {time.time()-t0:.1f}s')

del model, processor; gc.collect()
!du -sh {MERGED_DIR}

## 3 · Push merged model to HF

In [ ]:
from huggingface_hub import HfApi, create_repo
api = HfApi()
create_repo(REPO_MERGED, repo_type='model', private=True, exist_ok=True)
api.upload_folder(folder_path=MERGED_DIR, repo_id=REPO_MERGED, repo_type='model',
                  commit_message='DressApp Eyes — Phase 6 (Gemma 4 E2B LoRA merged)')
print('Uploaded to https://huggingface.co/' + REPO_MERGED)

## 4 · Build llama.cpp + convert to GGUF

In [ ]:
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git /content/llama.cpp || true
!pip install -q -r /content/llama.cpp/requirements.txt
!cmake -B /content/llama.cpp/build /content/llama.cpp -DGGML_CUDA=OFF -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF > /tmp/cm.log 2>&1 && echo cmake OK
!cmake --build /content/llama.cpp/build --config Release -j$(nproc) --target llama-quantize > /tmp/bld.log 2>&1 && echo build OK
!ls -l /content/llama.cpp/build/bin/llama-quantize

In [ ]:
# GGUF F16 conversion (Gemma 4 is supported in current llama.cpp main branch)
!python3 /content/llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outfile {GGUF_DIR}/phase6-f16.gguf --outtype f16
# Quantise to Q4_K_M (~2.5 GB, the edge-target quant)
!/content/llama.cpp/build/bin/llama-quantize {GGUF_DIR}/phase6-f16.gguf {GGUF_DIR}/phase6-Q4_K_M.gguf Q4_K_M
!ls -lh {GGUF_DIR}

## 5 · Push GGUFs

In [ ]:
create_repo(REPO_GGUF, repo_type='model', private=True, exist_ok=True)
api.upload_folder(folder_path=GGUF_DIR, repo_id=REPO_GGUF, repo_type='model',
                  commit_message='DressApp Eyes — Phase 6 GGUF (f16 + Q4_K_M)')
print('Uploaded to https://huggingface.co/' + REPO_GGUF)
print('\nWhen this cell is done, paste these repo IDs back to Neo:')
print('  merged:', REPO_MERGED)
print('  gguf:  ', REPO_GGUF)